In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

# Set working directory
base_dir = os.path.abspath("")
tokenized_data_path = os.path.join(base_dir, "tokenize_job", "tokenized_output", "tokenized_multi.jsonl")
output_dir = os.path.join(base_dir, "model_output")
os.makedirs(output_dir, exist_ok=True)

# Load tokenizer
tokenizer_dir = os.path.join(base_dir, "tokenize_job", "final_tokenizer")
tokenizer = BertTokenizer.from_pretrained(tokenizer_dir, local_files_only=True)
vocab_size = len(tokenizer)
print(f"✅ Tokenizer vocab size: {vocab_size}")

# Load tokenized dataset
tokenized_data = []
with open(tokenized_data_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line.strip())
        tokenized_data.append(obj)
print(f"✅ Loaded {len(tokenized_data)} samples")

# Labels
label2id = {"Rejected": 0, "Accepted": 1}
id2label = {v: k for k, v in label2id.items()}

# Dataset
class LegalDataset(Dataset):
    def __init__(self, data, max_length=512):
        self.data = data
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        token_ids = item["chunks"][0]["token_ids"]
        label = label2id.get(item.get("label", "Rejected"), 0)

        # Clip token_ids within vocab range
        token_ids = [min(id, vocab_size - 1) for id in token_ids[:self.max_length]]

        # Pad sequence
        pad_len = self.max_length - len(token_ids)
        input_ids = token_ids + [0] * pad_len
        attention_mask = [1 if t != 0 else 0 for t in input_ids]
 
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Split
dataset = LegalDataset(tokenized_data)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
print(f"✅ Train: {len(train_dataset)}, Val: {len(val_dataset)}")


In [ ]:
import torch.nn as nn
import copy

# Custom Encoder Layer
class CustomEncoderLayer(nn.Module):
    def __init__(self, d_model=768, n_heads=8, d_ff=3072, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_output, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.ff(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

# MoE Model
class NyayaNetMoE(nn.Module):
    def __init__(self, vocab_size, num_experts=2, num_labels=2, d_model=768, n_heads=8, d_ff=3072, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        base_layers = nn.ModuleList([CustomEncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)])
        self.experts = nn.ModuleList([copy.deepcopy(base_layers) for _ in range(num_experts)])
        self.classifier = nn.Linear(d_model, num_labels)
        self.dropout = nn.Dropout(dropout)
        self.num_experts = num_experts

    def forward(self, input_ids, attention_mask=None, court_ids=None):
        x = self.embedding(input_ids)
        key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        batch_size = input_ids.size(0)

        # Assign samples to experts
        if court_ids is None:
            court_ids = torch.randint(0, self.num_experts, (batch_size,), device=input_ids.device)

        # Collect expert outputs
        outputs = [None] * batch_size
        for i, expert in enumerate(self.experts):
            idxs = (court_ids == i).nonzero(as_tuple=False).squeeze(-1)
            if len(idxs) == 0:
                continue
            x_sub = x.index_select(0, idxs)
            mask_sub = key_padding_mask.index_select(0, idxs) if key_padding_mask is not None else None
            for layer in expert:
                x_sub = layer(x_sub, key_padding_mask=mask_sub)
            for j, bidx in enumerate(idxs):
                outputs[bidx] = x_sub[j]

        combined = torch.stack(outputs, dim=0)  # (batch, seq_len, d_model)

        # Use CLS token directly for classification
        cls_repr = combined[:, 0, :]
        logits = self.classifier(self.dropout(cls_repr))
        return logits

# Init model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = NyayaNetMoE(vocab_size=vocab_size, num_experts=2, num_labels=2, num_layers=2).to(device)
print("✅ Model initialized on", device)


In [ ]:
from tqdm import tqdm
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss()
num_epochs = 2

for epoch in range(num_epochs):
    # ---- Train ----
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attn_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f}")

    # ---- Validation ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attn_mask)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    print(f"Validation Accuracy: {acc:.2f}%")

# Save
save_path = os.path.join(output_dir, "nyaya_net_moe.pth")
torch.save(model.state_dict(), save_path)
print(f"✅ Model saved at {save_path}")
